<b><u>data set link: </b></u><a>https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata</a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
movies = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')


In [ ]:
movies.head()

In [ ]:
credits.head()

In [ ]:
movies = movies.merge(credits,on='title')

In [ ]:
movies.head(1)

In [ ]:
# imp columns: 
# genres 
# id 
# keywords
# title
# overview 
# cast 
# crew

movies = movies[['id','genres','keywords','title','overview','cast','crew']]

In [ ]:
movies.head(1)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace= True)

In [ ]:
movies.isnull().sum()

In [ ]:
print('duplicate:',movies.duplicated().sum())

In [ ]:
movies.iloc[0].genres

In [ ]:
# [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
 # we want like this : ['Action','Advanture','Fantacy','Scince Fiction']

In [ ]:
import ast 

def convert(obj):
    L = []
    for o in ast.literal_eval(obj):
        L.append(o['name'])
    return L

In [ ]:

ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
    c = 0
    L = []
    for i in ast.literal_eval(obj):
        if c != 3:
            L.append(i['name'])
            c+=1
        else:
            break
    return L

In [ ]:
movies['cast']= movies['cast'].apply(convert3)

In [ ]:
def fetch_director(obj):
    L = []
    for o in ast.literal_eval(obj):
        if o['job'] == 'Director':
            L.append(o['name'])
            break
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" " ,"") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" " ,"") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" " ,"") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" " ,"") for i in x])

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew'] 

In [ ]:
movies.head()

In [ ]:
df = movies[['id','title','tags']]
df.head()

In [ ]:
df['tags'] = df['tags'].apply(lambda x:" ".join(x))

In [ ]:
df['tags'][0]

In [ ]:
df['tags'] = df['tags'].apply(lambda x:x.lower())

In [ ]:
df.head()

<b><u>55:27 concept of vectorization 
that  we will use in this prject
link of video:</b></u> <a>https://youtu.be/1xtrIEwY_zY?si=xdRhXoPZB65jlnqu</a>


steming
['loved','loving','love'] =>['love','love','love']

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [ ]:
df['tags'] = df['tags'].apply(stem)

In [ ]:
# example
ps.stem("loving")

In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer 
cv = CountVectorizer(max_features= 5000,stop_words='english')

In [ ]:
vectors = cv.fit_transform(df['tags']).toarray()

In [ ]:
vectors

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

In [ ]:
#  find cosine distance for each movie with all 
# calculate angles between two vectors 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
cosine_similarity(vectors).shape

In [ ]:
list(enumerate(similarity[0]))

In [ ]:
sorted(list(enumerate(similarity[0])),reverse= True,key =lambda x:x[1])

# [(0, np.float64(1.000000000000002))
# this is the similarity of movie with itself

In [ ]:
sorted(list(enumerate(similarity[0])),reverse= True,key =lambda x:x[1])[1:6]


In [ ]:
def recomend(movie):
    movie_index = df[df['title'] == movie ].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse= True,key =lambda x:x[1])[1:6]

    for i in movies_list:
        print(df.iloc[i[0]].title)

In [ ]:
recomend('Batman Begins')

In [ ]:
df.iloc[1214].title

In [ ]:
# app integration 

In [ ]:
import pickle

In [ ]:
# pickle.dump(df,open('movies.pkl','wb'))

with open("/kaggle/working/movies_dict.pkl", "wb") as f:
    pickle.dump(df.to_dict(), f)

In [ ]:
# pickle.dump(similarity,open('similarity.pkl','wb'))

with open("/kaggle/working/similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)